In [1]:
from __future__ import annotations
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Dict, List, Optional, Any, Union, Annotated, Literal
import random
import yaml
from collections import defaultdict, deque
from pathlib import Path
import scoop


In [2]:
from pathlib import Path
from scoop.utils import load_transformation_graph, make_mermaid
from scoop.configuration import transformation_graph

In [3]:
transformation_graph

TransformationGraph(nodes={'person.age': NumericNode(name='person.age', type='numeric', built_distribution=NumericDistribution(name='person.age', mean=70.0, sd=7.0, min_value=50.0, max_value=95.0, as_int=True), active_when=None, mean=70.0, sd=7.0, min_value=50.0, max_value=95.0, as_int=True), 'person.age_group': NumericRangeNode(name='person.age_group', type='numeric_range', built_distribution=None, active_when=None, from_='person.age', ranges=['<60', '60-69', '70-79', '80+']), 'person.year_of_diagnosis': NumericNode(name='person.year_of_diagnosis', type='numeric', built_distribution=NumericDistribution(name='person.year_of_diagnosis', mean=0.0, sd=None, min_value=2015.0, max_value=2020.0, as_int=True), active_when=None, mean=0, sd=None, min_value=2015.0, max_value=2020.0, as_int=True), 'person.years_since_diagnosis': NumericNode(name='person.years_since_diagnosis', type='numeric', built_distribution=NumericDistribution(name='person.years_since_diagnosis', mean=0.0, sd=None, min_value=

In [4]:
# graph = load_transformation_graph(config_path / "probabilistic_transformations.yaml")
print(transformation_graph.order)          # dependency order
print(transformation_graph.nodes)          # correct node types
print(transformation_graph.edges)          # CPT edges
print(transformation_graph.sample_one())   # fully conditional sampling


['person.age', 'person.age_group', 'person.year_of_diagnosis', 'person.years_since_diagnosis', 'person.sex', 'person.smoking_history', 'person.variant', 'person.weight', 'condition_occurrence.laterality', 'condition_episode.classification:', 'condition_episode.group_stage', 'condition_episode.targetable_mutation', 'treatment_episode.neoadjuvant_targeted', 'treatment_episode.definitive_sbrt', 'treatment_episode.surgery_port', 'person.performance_status', 'condition_occurrence.cancer_stage', 'condition_occurrence.morphology', 'condition_occurrence.lobe', 'condition_episode.radiotherapy_eligibility', 'condition_episode.resectability']
{'person.age': NumericNode(name='person.age', type='numeric', built_distribution=NumericDistribution(name='person.age', mean=70.0, sd=7.0, min_value=50.0, max_value=95.0, as_int=True), active_when=None, mean=70.0, sd=7.0, min_value=50.0, max_value=95.0, as_int=True), 'person.age_group': NumericRangeNode(name='person.age_group', type='numeric_range', built_di

In [5]:
print(make_mermaid(transformation_graph))

flowchart LR

    subgraph person
        person_weight["person.weight<br/>N(80.0, 15.0)"]:::node_person_weight
        classDef node_person_weight fill:#A3C4FF,stroke:#333,stroke-width:1px;
        person_year_of_diagnosis["person.year_of_diagnosis<br/>N(0, None)"]:::node_person_year_of_diagnosis
        classDef node_person_year_of_diagnosis fill:#A3C4FF,stroke:#333,stroke-width:1px;
        person_age["person.age<br/>N(70.0, 7.0)"]:::node_person_age
        classDef node_person_age fill:#A3C4FF,stroke:#333,stroke-width:1px;
        person_age_group["person.age_group<br/><60, 60-69, 70-79, 80+"]:::node_person_age_group
        classDef node_person_age_group fill:#E6C3F9,stroke:#333,stroke-width:1px;
        person_years_since_diagnosis["person.years_since_diagnosis<br/>N(0, None)"]:::node_person_years_since_diagnosis
        classDef node_person_years_since_diagnosis fill:#A3C4FF,stroke:#333,stroke-width:1px;
        person_sex["person.sex<br/>male, female"]:::node_person_sex
       

In [ ]:
{k: v for k, v in transformation_graph.nodes.items() if getattr(v, "active_when", None)}

In [ ]:
activation_edges = []
for name, node in transformation_graph.nodes.items():
    active_when = getattr(node, "active_when", None)
    if not active_when:
        continue

    for upstream in active_when.keys():
        if upstream not in transformation_graph.nodes:
            raise ValueError(
                f"Node '{name}' has active_when dependency on unknown node '{upstream}'"
            )

        activation_edges.append(
            (
                upstream,
                name,
                "active",
                None,
                "activation",   # edge kind
            )
        )

In [ ]:
activation_edges

In [ ]:
active_when